# Session 11: Applications — quantum mechanics, fluid dynamics, and reaction–diffusion systems

Sessions 1–10 built the full toolkit for physics-informed neural networks (PINNs): the mathematical foundations, the PyTorch implementation, inverse problems, loss balancing, advanced sampling strategies, and domain decomposition. This session and [Session 12](Session12.ipynb) apply that toolkit to three physically distinct problems that span quantum mechanics, fluid dynamics, and nonlinear pattern formation.

Working through these examples will show you that the PINN recipe does not change between application domains — only the partial differential equations (PDEs) and boundary conditions do. The three problems are:

1. **The time-independent Schrödinger equation** — an eigenvalue problem in quantum mechanics, solved here as an inverse problem in which the ground-state energy $E$ is a learnable parameter.
2. **The Stokes (creeping-flow) vorticity equation** — the low-Reynolds-number limit of the Navier–Stokes equations for steady flow in a 2D cavity, which reduces to a Laplace equation framed in fluid-mechanics terms.
3. **The Allen–Cahn equation** — a 1D reaction–diffusion equation whose solutions exhibit phase-interface dynamics, solved by a PINN and validated against a finite-difference (FD) reference.

**Prerequisites**: [Session 8](Session8.ipynb) (inverse problems and 2D Poisson), [Session 9](Session9.ipynb) (loss balancing), and [Session 10](Session10.ipynb) (advanced topics).

## Learning objectives

By the end of this session you will be able to:

- Formulate the time-independent Schrödinger equation as a PINN eigenvalue problem.
- Recognise when a Navier–Stokes problem reduces to a simpler PDE and exploit that structure.
- Implement and validate a PINN for a nonlinear reaction–diffusion equation using a finite-difference reference solution.

## 1. Common imports and device setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. The Schrödinger equation: ground state of the infinite square well

### 2.1 Physical background

The time-independent Schrödinger equation (TISE) is the central eigenvalue problem of non-relativistic quantum mechanics. In one spatial dimension it reads:

$$
-\frac{\hbar^2}{2m}\,\psi''(x) + V(x)\,\psi(x) = E\,\psi(x)
$$

where $\psi(x)$ is the wave function, $V(x)$ the potential energy, and $E$ the (unknown) eigenenergy. We adopt natural units $\hbar = m = 1$, which simplifies the equation to:

$$
-\frac{1}{2}\psi''(x) + V(x)\,\psi(x) = E\,\psi(x), \quad x \in [0, 1]
$$

**The infinite square well** sets $V(x) = 0$ inside $[0,1]$ and $V = \infty$ outside, which imposes Dirichlet boundary conditions $\psi(0) = \psi(1) = 0$. The exact normalised ground-state solution is:

$$
\psi_1(x) = \sqrt{2}\,\sin(\pi x), \qquad E_1 = \frac{\pi^2}{2} \approx 4.9348
$$

### 2.2 Formulation as a PINN inverse problem

The eigenenergy $E$ is unknown — this makes the TISE an **inverse problem** of exactly the type studied in [Session 8](Session8.ipynb). We treat $E$ as a learnable `nn.Parameter` alongside the network weights $\theta$, and the network $\psi_\theta(x)$ approximates the wave function.

The loss has three components:

$$
\mathcal{L} = \mathcal{L}_{\text{PDE}} + \lambda_{\text{BC}}\,\mathcal{L}_{\text{BC}} + \lambda_{\text{norm}}\,\mathcal{L}_{\text{norm}}
$$

| Term | Expression | Purpose |
|---|---|---|
| $\mathcal{L}_{\text{PDE}}$ | $\frac{1}{N_c}\sum\lvert -\tfrac{1}{2}\psi'' - E\psi \rvert^2$ | Enforces the TISE residual |
| $\mathcal{L}_{\text{BC}}$ | $\psi(0)^2 + \psi(1)^2$ | Dirichlet boundary conditions |
| $\mathcal{L}_{\text{norm}}$ | $\bigl(\int_0^1 \psi^2\,dx - 1\bigr)^2$ | Normalisation $\langle\psi\vert\psi\rangle = 1$ |

Without the normalisation constraint the trivial solution $\psi \equiv 0$ minimises $\mathcal{L}_{\text{PDE}}$ and $\mathcal{L}_{\text{BC}}$ exactly. Adding $\mathcal{L}_{\text{norm}}$ rules out this degenerate minimum. The integral is approximated by a sum over collocation points.

**Sign ambiguity.** The TISE is linear, so if $\psi$ is a solution then $-\psi$ is also a solution. The PINN may converge to either sign; we fix the sign at the end by ensuring the midpoint value $\psi(0.5)$ is positive.

### 2.3 Model definition

In [ ]:
class SchrodingerPINN(nn.Module):
    """PINN for the 1D time-independent Schrödinger equation.

    The network maps x -> psi(x).  The eigenenergy E is a learnable
    parameter, updated jointly with the network weights.
    """

    def __init__(self, layers=None):
        super().__init__()
        if layers is None:
            layers = [1, 64, 64, 64, 1]
        seq = []
        for i in range(len(layers) - 2):
            seq += [nn.Linear(layers[i], layers[i + 1]), nn.Tanh()]
        seq.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*seq)
        # Initialise E near the true value so training is tractable;
        # in a real problem one would start further away.
        self.log_E = nn.Parameter(torch.log(torch.tensor(3.0)))

    def forward(self, x):
        return self.net(x)

    @property
    def E(self):
        """Energy eigenvalue, constrained positive via log-parameterisation."""
        return torch.exp(self.log_E)

### 2.4 Collocation and boundary points

In [ ]:
N_col_qm = 2000
torch.manual_seed(0)

# Interior collocation points
x_col = torch.rand(N_col_qm, 1, device=device).requires_grad_(True)

# Boundary points
x_bc = torch.tensor([[0.0], [1.0]], device=device)

# Dense evaluation grid for loss integrals and plotting
x_plot = torch.linspace(0, 1, 500, device=device).unsqueeze(1)

### 2.5 Training

In [ ]:
schrodinger_model = SchrodingerPINN().to(device)
schrodinger_opt = torch.optim.Adam(schrodinger_model.parameters(), lr=1e-3)

EPOCHS_QM = 15000
lambda_bc   = 100.0   # strong BC enforcement
lambda_norm = 50.0    # normalisation constraint

E_history  = []
loss_history_qm = []

for epoch in range(EPOCHS_QM):
    schrodinger_opt.zero_grad()

    # --- PDE residual: -1/2 * psi'' - E * psi = 0 ---
    psi = schrodinger_model(x_col)
    dpsi = torch.autograd.grad(
        psi, x_col,
        grad_outputs=torch.ones_like(psi),
        create_graph=True, retain_graph=True
    )[0]
    d2psi = torch.autograd.grad(
        dpsi, x_col,
        grad_outputs=torch.ones_like(dpsi),
        create_graph=True, retain_graph=True
    )[0]

    residual = -0.5 * d2psi - schrodinger_model.E * psi
    loss_pde = torch.mean(residual ** 2)

    # --- Boundary conditions: psi(0) = psi(1) = 0 ---
    psi_bc = schrodinger_model(x_bc)
    loss_bc = torch.mean(psi_bc ** 2)

    # --- Normalisation: integral psi^2 dx ≈ 1 ---
    with torch.no_grad():
        psi_grid = schrodinger_model(x_plot)
        norm_val = torch.trapezoid(psi_grid.squeeze() ** 2,
                                   x_plot.squeeze())
    # Re-evaluate with grad to allow differentiation through norm_val
    psi_grid_g = schrodinger_model(x_plot)
    norm_val_g = torch.trapezoid(psi_grid_g.squeeze() ** 2,
                                  x_plot.squeeze())
    loss_norm = (norm_val_g - 1.0) ** 2

    loss = loss_pde + lambda_bc * loss_bc + lambda_norm * loss_norm
    loss.backward()
    schrodinger_opt.step()

    E_history.append(schrodinger_model.E.item())
    loss_history_qm.append(loss.item())

    if epoch % 3000 == 0:
        print(f"Epoch {epoch:6d} | Loss: {loss.item():.3e} | "
              f"PDE: {loss_pde.item():.3e} | BC: {loss_bc.item():.3e} | "
              f"Norm: {loss_norm.item():.3e} | E: {schrodinger_model.E.item():.5f}")

print(f"\nExact E_1  = {np.pi**2 / 2:.6f}")
print(f"Inferred E = {schrodinger_model.E.item():.6f}")

### 2.6 Results

In [ ]:
with torch.no_grad():
    x_np = x_plot.cpu().numpy().flatten()
    psi_pinn = schrodinger_model(x_plot).cpu().numpy().flatten()

# Fix sign so that psi(0.5) > 0
mid_idx = len(x_np) // 2
if psi_pinn[mid_idx] < 0:
    psi_pinn = -psi_pinn

psi_exact = np.sqrt(2) * np.sin(np.pi * x_np)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Wave function comparison
axes[0].plot(x_np, psi_exact, 'k--', lw=2, label='Exact $\\psi_1$')
axes[0].plot(x_np, psi_pinn,  'C0',  lw=2, label='PINN $\\psi$')
axes[0].set_xlabel('$x$')
axes[0].set_ylabel('$\\psi(x)$')
axes[0].set_title('Wave function: PINN vs exact')
axes[0].legend()

# Absolute error
axes[1].plot(x_np, np.abs(psi_pinn - psi_exact), 'C1', lw=1.5)
axes[1].set_xlabel('$x$')
axes[1].set_ylabel('$|\\psi_{\\mathrm{PINN}} - \\psi_{\\mathrm{exact}}|$')
axes[1].set_title('Point-wise absolute error')

# Convergence of E
axes[2].plot(E_history, lw=1.5, label='Inferred $E$')
axes[2].axhline(np.pi**2 / 2, color='r', ls='--', lw=2,
                label=f'Exact $E_1 = \\pi^2/2 \\approx {np.pi**2/2:.4f}$')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('$E$')
axes[2].set_title('Convergence of inferred energy')
axes[2].legend()

plt.tight_layout()
plt.show()

rel_err_psi = np.sqrt(np.mean((psi_pinn - psi_exact)**2)) / np.sqrt(np.mean(psi_exact**2))
rel_err_E   = abs(schrodinger_model.E.item() - np.pi**2 / 2) / (np.pi**2 / 2)
print(f'Relative L2 error in psi: {rel_err_psi:.4e}')
print(f'Relative error in E:      {rel_err_E:.4e}')

### 2.7 Discussion

The PINN successfully recovers both the wave function shape and the eigenenergy. Key observations:

- **Inverse nature**: $E$ was not prescribed — it emerged from minimising the combined loss, exactly as the viscosity $\nu$ emerged in [Session 8](Session8.ipynb).
- **Normalisation constraint**: without $\mathcal{L}_{\text{norm}}$, the optimiser finds the trivial solution $\psi \equiv 0$. The normalisation term breaks this degeneracy at the cost of slightly slower convergence.
- **Sign degeneracy**: the linearity of the TISE means the sign of $\psi$ is not determined by the PDE alone. We fixed it post hoc; a more principled approach is to add a soft constraint such as $\psi(0.5) > 0$.
- **Higher modes**: to find the first excited state $\psi_2$, one must add an orthogonality constraint $\int \psi_1 \psi_2\,dx = 0$. This is a natural extension for the project in [Session 12](Session12.ipynb).

## 3. Simplified Navier–Stokes: Stokes flow in the lid-driven cavity

### 3.1 Physical background

The incompressible Navier–Stokes (NS) equations govern the motion of viscous fluids. In 2D, the vorticity–streamfunction formulation replaces the velocity field $(u, v)$ with the vorticity $\omega = \partial v/\partial x - \partial u/\partial y$ and the streamfunction $\psi$ (related to velocity by $u = \partial\psi/\partial y$, $v = -\partial\psi/\partial x$). The steady NS equation in vorticity form is:

$$
\nabla^2 \omega = \mathrm{Re}\left(u\frac{\partial\omega}{\partial x} + v\frac{\partial\omega}{\partial y}\right)
$$

where $\mathrm{Re} = UL/\nu$ is the Reynolds number. In the **Stokes (creeping-flow) limit** $\mathrm{Re} \to 0$, the nonlinear convective term on the right-hand side vanishes, leaving:

$$
\nabla^2 \omega = 0 \quad \Longleftrightarrow \quad \frac{\partial^2\omega}{\partial x^2} + \frac{\partial^2\omega}{\partial y^2} = 0
$$

This is simply the **Laplace equation**, the homogeneous version of the Poisson equation you solved in [Session 8](Session8.ipynb).

**Lid-driven cavity geometry.** The domain is the unit square $[0,1]^2$. Three walls are stationary ($\omega = 0$); the top lid moves with unit velocity, imposing $\omega = 1$ on $y = 1$ (a simplified Dirichlet condition replacing the physical $\omega = -2 U/h$ wall boundary condition at low Re).

| Boundary | Condition |
|---|---|
| Bottom ($y=0$) | $\omega = 0$ |
| Left ($x=0$) | $\omega = 0$ |
| Right ($x=1$) | $\omega = 0$ |
| Lid ($y=1$) | $\omega = 1$ |

**Connection to Session 8.** This problem is structurally identical to the 2D Poisson equation in [Session 8](Session8.ipynb) with $f = 0$, but with non-homogeneous boundary conditions on one side. It demonstrates how the same PINN architecture adapts to a new physical context by a straightforward change of boundary condition.

### 3.2 Model and collocation points

In [ ]:
class StokesPINN(nn.Module):
    """MLP for the 2D Stokes vorticity equation: input (x, y), output omega."""

    def __init__(self, layers=None):
        super().__init__()
        if layers is None:
            layers = [2, 64, 64, 64, 1]
        seq = []
        for i in range(len(layers) - 2):
            seq += [nn.Linear(layers[i], layers[i + 1]), nn.Tanh()]
        seq.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*seq)

    def forward(self, xy):
        return self.net(xy)


N_col_st = 8000
N_bc_st  = 300
torch.manual_seed(1)

# Interior collocation
col_st = torch.rand(N_col_st, 2, device=device).requires_grad_(True)

# Boundary points: parametrised by s in [0,1]
s = torch.rand(N_bc_st, device=device)
z = torch.zeros(N_bc_st, device=device)
o = torch.ones(N_bc_st, device=device)

bc_bottom = torch.stack([s, z], dim=1)    # y = 0, omega = 0
bc_left   = torch.stack([z, s], dim=1)    # x = 0, omega = 0
bc_right  = torch.stack([o, s], dim=1)    # x = 1, omega = 0
bc_lid    = torch.stack([s, o], dim=1)    # y = 1, omega = 1

bc_walls    = torch.cat([bc_bottom, bc_left, bc_right], dim=0).to(device)
bc_lid_pts  = bc_lid.to(device)

### 3.3 Training

In [ ]:
stokes_model = StokesPINN().to(device)
stokes_opt = torch.optim.Adam(stokes_model.parameters(), lr=1e-3)

EPOCHS_ST = 10000
lambda_bc_st = 10.0

loss_history_st = []

for epoch in range(EPOCHS_ST):
    stokes_opt.zero_grad()

    # PDE: Laplacian(omega) = 0
    omega = stokes_model(col_st)
    grads = torch.autograd.grad(
        omega, col_st,
        grad_outputs=torch.ones_like(omega),
        create_graph=True, retain_graph=True
    )[0]

    omega_xx = torch.autograd.grad(
        grads[:, 0:1], col_st,
        grad_outputs=torch.ones_like(grads[:, 0:1]),
        create_graph=True, retain_graph=True
    )[0][:, 0:1]

    omega_yy = torch.autograd.grad(
        grads[:, 1:2], col_st,
        grad_outputs=torch.ones_like(grads[:, 1:2]),
        create_graph=True, retain_graph=True
    )[0][:, 1:2]

    loss_pde_st = torch.mean((omega_xx + omega_yy) ** 2)

    # Wall BCs: omega = 0 on three sides
    loss_walls = torch.mean(stokes_model(bc_walls) ** 2)

    # Lid BC: omega = 1 on y = 1
    loss_lid = torch.mean((stokes_model(bc_lid_pts) - 1.0) ** 2)

    loss_st = loss_pde_st + lambda_bc_st * (loss_walls + loss_lid)
    loss_st.backward()
    stokes_opt.step()

    loss_history_st.append(loss_st.item())

    if epoch % 2000 == 0:
        print(f"Epoch {epoch:6d} | Loss: {loss_st.item():.3e} | "
              f"PDE: {loss_pde_st.item():.3e} | "
              f"Walls: {loss_walls.item():.3e} | "
              f"Lid: {loss_lid.item():.3e}")

### 3.4 Visualisation of the vorticity and streamfunction fields

In [ ]:
Ng_st = 80
xi_st = torch.linspace(0, 1, Ng_st)
yi_st = torch.linspace(0, 1, Ng_st)
Xi_st, Yi_st = torch.meshgrid(xi_st, yi_st, indexing='ij')
XY_st = torch.stack([Xi_st.flatten(), Yi_st.flatten()], dim=1).to(device)

with torch.no_grad():
    omega_grid = stokes_model(XY_st).cpu().numpy().reshape(Ng_st, Ng_st)

X_np = Xi_st.numpy()
Y_np = Yi_st.numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Vorticity field
c0 = axes[0].contourf(X_np, Y_np, omega_grid, levels=40, cmap='RdBu_r')
plt.colorbar(c0, ax=axes[0], label='$\\omega$')
axes[0].set_title('Vorticity field $\\omega(x, y)$')
axes[0].set_xlabel('$x$')
axes[0].set_ylabel('$y$')
axes[0].set_aspect('equal')

# Training loss
axes[1].semilogy(loss_history_st, lw=1.5, color='C2')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Total loss')
axes[1].set_title('Training loss — Stokes cavity')

plt.tight_layout()
plt.show()

### 3.5 Discussion

The vorticity field shows the characteristic pattern of Stokes flow in a lid-driven cavity: vorticity is large near the moving lid and decays towards the stationary walls. Key points:

- **Reuse of architecture**: the `StokesPINN` is structurally identical to the `PoissonPINN` from [Session 8](Session8.ipynb) — the only differences are the source term (zero here) and the non-homogeneous boundary condition on the lid.
- **Fluid mechanics interpretation**: the vorticity $\omega$ quantifies local rotation in the flow. The lid imparts positive vorticity; this diffuses into the interior via the Laplacian term.
- **Limitations of the Stokes approximation**: real cavity flows at finite Reynolds number have a primary recirculating vortex and, at higher Re, secondary corner vortices. Including the convective term $\mathrm{Re}(\mathbf{u}\cdot\nabla)\omega$ would require a more elaborate PINN with the streamfunction as a second network output.
- **Comparison with Session 8**: the 2D Poisson equation in [Session 8](Session8.ipynb) had a non-trivial source term and homogeneous BCs; here the source is zero but the BCs are non-homogeneous. Both are handled within the same PINN framework.

## 4. The Allen–Cahn equation: reaction–diffusion and phase interfaces

### 4.1 Physical background

The Allen–Cahn equation is a prototypical **reaction–diffusion** PDE that models phase separation in materials science and pattern formation in biological systems. In one spatial dimension on the domain $[0,1] \times [0, T]$ it reads:

$$
\frac{\partial u}{\partial t} = \varepsilon^2\,\frac{\partial^2 u}{\partial x^2} + u(1 - u^2)
$$

where:
- $u(x, t)$ is the order parameter (e.g., local phase composition), taking values in $[-1, 1]$.
- $\varepsilon$ controls the interface width: small $\varepsilon$ gives sharp interfaces between the stable phases $u = \pm 1$.
- The nonlinear reaction term $u(1-u^2)$ drives the field towards the minima of the double-well potential $F(u) = \frac{1}{4}(u^2-1)^2$.

**Why this equation matters.** The Allen–Cahn equation is a canonical example of a stiff, nonlinear PDE. The nonlinear term can cause the solution to develop sharp fronts that classical PINN methods struggle to resolve — a practical demonstration of the spectral bias discussed in [Session 9](Session9.ipynb).

**Setup for this session.** We use:
- $\varepsilon = 0.1$, $T = 1$, domain $x \in [0, 1]$.
- Initial condition: $u(x, 0) = x^2\cos(\pi x)$.
- Neumann boundary conditions: $\partial u/\partial x\big|_{x=0} = \partial u/\partial x\big|_{x=1} = 0$.
- A reference solution is computed by a finite-difference (FD) method to validate the PINN.

### 4.2 Finite-difference reference solution

We use an implicit Crank–Nicolson scheme for the diffusion term and explicit treatment of the nonlinear reaction term (IMEX). This gives a stable reference solution on a fine grid.

In [ ]:
# -----------------------------------------------------------------
# Finite-difference reference: Crank-Nicolson for diffusion,
# explicit (forward Euler) for the nonlinear reaction term.
# -----------------------------------------------------------------

eps_ac = 0.1
T_ac   = 1.0
Nx_fd  = 256
Nt_fd  = 5000

dx = 1.0 / (Nx_fd - 1)
dt = T_ac / Nt_fd

x_fd = np.linspace(0, 1, Nx_fd)
u_fd = x_fd**2 * np.cos(np.pi * x_fd)   # initial condition

# Crank-Nicolson matrix for the diffusion operator with Neumann BCs
# (zero-flux: ghost-point approach, so the end rows use one-sided coefficients)
r = eps_ac**2 * dt / (2.0 * dx**2)

diag_main  = (1 + 2 * r) * np.ones(Nx_fd)
diag_off   = -r * np.ones(Nx_fd - 1)

# Neumann BC: du/dx = 0 at x=0 and x=1 → ghost-point symmetry
diag_main[0]  = 1 + r
diag_main[-1] = 1 + r

A_upper = np.diag(diag_main) + np.diag(diag_off, 1) + np.diag(diag_off, -1)

diag_rhs_main = (1 - 2 * r) * np.ones(Nx_fd)
diag_rhs_main[0]  = 1 - r
diag_rhs_main[-1] = 1 - r
A_lower = (np.diag(diag_rhs_main)
           + np.diag(r * np.ones(Nx_fd - 1), 1)
           + np.diag(r * np.ones(Nx_fd - 1), -1))

# Pre-factorise the left-hand-side matrix for efficiency
from numpy.linalg import solve as np_solve

# Store every 50th time step for visualisation
save_every = 50
u_fd_history = [u_fd.copy()]
t_fd_history = [0.0]

for n in range(Nt_fd):
    reaction = u_fd * (1.0 - u_fd**2)
    rhs = A_lower @ u_fd + dt * reaction
    u_fd = np_solve(A_upper, rhs)
    u_fd = np.clip(u_fd, -1.5, 1.5)   # safeguard against numerical blow-up
    if (n + 1) % save_every == 0:
        u_fd_history.append(u_fd.copy())
        t_fd_history.append((n + 1) * dt)

u_fd_history = np.array(u_fd_history)   # shape: (Nt_saved, Nx_fd)
t_fd_history = np.array(t_fd_history)

print(f'FD solution computed: {u_fd_history.shape[0]} snapshots, '
      f'final time = {t_fd_history[-1]:.3f}')

# Quick check: u should remain in [-1.5, 1.5]
print(f'u range: [{u_fd_history.min():.4f}, {u_fd_history.max():.4f}]')

In [ ]:
# Visualise the FD reference solution as a space-time heatmap
T_grid, X_grid = np.meshgrid(t_fd_history, x_fd)

fig, ax = plt.subplots(figsize=(9, 5))
c = ax.pcolormesh(T_grid, X_grid, u_fd_history.T,
                  cmap='RdBu_r', shading='auto', vmin=-1, vmax=1)
plt.colorbar(c, ax=ax, label='$u$')
ax.set_xlabel('$t$')
ax.set_ylabel('$x$')
ax.set_title('Allen–Cahn reference solution (finite differences)')
plt.tight_layout()
plt.show()

### 4.3 PINN for the Allen–Cahn equation

The network takes $(x, t)$ as input and outputs $u_\theta(x, t)$. The loss is:

$$
\mathcal{L} = \mathcal{L}_{\text{PDE}} + \lambda_{\text{IC}}\,\mathcal{L}_{\text{IC}} + \lambda_{\text{BC}}\,\mathcal{L}_{\text{BC}}
$$

where:
- $\mathcal{L}_{\text{PDE}} = \frac{1}{N_c}\sum\bigl\lvert u_t - \varepsilon^2 u_{xx} - u(1-u^2)\bigr\rvert^2$
- $\mathcal{L}_{\text{IC}} = \frac{1}{N_i}\sum\bigl\lvert u_\theta(x_i, 0) - x_i^2\cos(\pi x_i)\bigr\rvert^2$
- $\mathcal{L}_{\text{BC}} = \frac{1}{N_b}\sum\bigl\lvert u_x(0, t_j)\bigr\rvert^2 + \bigl\lvert u_x(1, t_k)\bigr\rvert^2$ (Neumann)

In [ ]:
class AllenCahnPINN(nn.Module):
    """MLP for the 1D Allen-Cahn equation: input (x, t), output u."""

    def __init__(self, layers=None):
        super().__init__()
        if layers is None:
            layers = [2, 64, 64, 64, 64, 1]
        seq = []
        for i in range(len(layers) - 2):
            seq += [nn.Linear(layers[i], layers[i + 1]), nn.Tanh()]
        seq.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*seq)

    def forward(self, xt):
        return self.net(xt)


# --- Collocation and boundary/initial condition points ---
N_col_ac = 10000
N_ic_ac  = 500
N_bc_ac  = 200
torch.manual_seed(2)

# Interior collocation: (x, t) in [0,1] x [0,1]
col_ac = torch.rand(N_col_ac, 2, device=device).requires_grad_(True)

# Initial condition: t = 0
ic_x_ac = torch.rand(N_ic_ac, 1, device=device)
ic_t_ac = torch.zeros(N_ic_ac, 1, device=device)
ic_xt_ac = torch.cat([ic_x_ac, ic_t_ac], dim=1)
u_ic_ac = (ic_x_ac ** 2) * torch.cos(np.pi * ic_x_ac)

# Neumann BCs: du/dx = 0 at x = 0 and x = 1
t_bc_ac = torch.rand(N_bc_ac, 1, device=device)
bc_left_ac  = torch.cat([torch.zeros(N_bc_ac, 1, device=device), t_bc_ac],
                         dim=1).requires_grad_(True)
bc_right_ac = torch.cat([torch.ones( N_bc_ac, 1, device=device), t_bc_ac],
                         dim=1).requires_grad_(True)

### 4.4 Training

In [ ]:
ac_model = AllenCahnPINN().to(device)
ac_opt = torch.optim.Adam(ac_model.parameters(), lr=1e-3)

EPOCHS_AC  = 20000
lambda_ic_ac = 20.0
lambda_bc_ac = 10.0

loss_history_ac = []

for epoch in range(EPOCHS_AC):
    ac_opt.zero_grad()

    # --- PDE residual: u_t - eps^2 * u_xx - u(1 - u^2) = 0 ---
    u_col = ac_model(col_ac)
    grads_col = torch.autograd.grad(
        u_col, col_ac,
        grad_outputs=torch.ones_like(u_col),
        create_graph=True, retain_graph=True
    )[0]
    u_x_col = grads_col[:, 0:1]
    u_t_col = grads_col[:, 1:2]

    u_xx_col = torch.autograd.grad(
        u_x_col, col_ac,
        grad_outputs=torch.ones_like(u_x_col),
        create_graph=True, retain_graph=True
    )[0][:, 0:1]

    residual_ac = u_t_col - eps_ac**2 * u_xx_col - u_col * (1.0 - u_col**2)
    loss_pde_ac = torch.mean(residual_ac ** 2)

    # --- Initial condition ---
    loss_ic_ac = torch.mean((ac_model(ic_xt_ac) - u_ic_ac) ** 2)

    # --- Neumann BCs: du/dx = 0 at x=0 and x=1 ---
    u_left  = ac_model(bc_left_ac)
    u_right = ac_model(bc_right_ac)

    du_dx_left = torch.autograd.grad(
        u_left, bc_left_ac,
        grad_outputs=torch.ones_like(u_left),
        create_graph=True, retain_graph=True
    )[0][:, 0:1]

    du_dx_right = torch.autograd.grad(
        u_right, bc_right_ac,
        grad_outputs=torch.ones_like(u_right),
        create_graph=True, retain_graph=True
    )[0][:, 0:1]

    loss_bc_ac = torch.mean(du_dx_left ** 2) + torch.mean(du_dx_right ** 2)

    loss_ac = loss_pde_ac + lambda_ic_ac * loss_ic_ac + lambda_bc_ac * loss_bc_ac
    loss_ac.backward()
    ac_opt.step()

    loss_history_ac.append(loss_ac.item())

    if epoch % 4000 == 0:
        print(f"Epoch {epoch:6d} | Loss: {loss_ac.item():.3e} | "
              f"PDE: {loss_pde_ac.item():.3e} | "
              f"IC: {loss_ic_ac.item():.3e} | "
              f"BC: {loss_bc_ac.item():.3e}")

### 4.5 Comparison with the finite-difference reference

In [ ]:
# Evaluate PINN on the same (x, t) grid used for the FD solution
Nt_vis  = len(t_fd_history)
Nx_vis  = len(x_fd)

x_vis = torch.tensor(x_fd, dtype=torch.float32)
t_vis = torch.tensor(t_fd_history, dtype=torch.float32)
X_vis, T_vis = torch.meshgrid(x_vis, t_vis, indexing='ij')
XT_vis = torch.stack([X_vis.flatten(), T_vis.flatten()], dim=1).to(device)

with torch.no_grad():
    u_pinn_ac = ac_model(XT_vis).cpu().numpy().reshape(Nx_vis, Nt_vis)

# u_fd_history has shape (Nt_vis, Nx_vis); transpose to (Nx_vis, Nt_vis)
u_fd_vis = u_fd_history.T

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

vmin, vmax = -1.0, 1.0

# FD reference
im0 = axes[0, 0].pcolormesh(T_vis.numpy(), X_vis.numpy(), u_fd_vis,
                              cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
plt.colorbar(im0, ax=axes[0, 0], label='$u$')
axes[0, 0].set_title('FD reference solution')
axes[0, 0].set_xlabel('$t$'); axes[0, 0].set_ylabel('$x$')

# PINN solution
im1 = axes[0, 1].pcolormesh(T_vis.numpy(), X_vis.numpy(), u_pinn_ac,
                              cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
plt.colorbar(im1, ax=axes[0, 1], label='$u$')
axes[0, 1].set_title('PINN solution')
axes[0, 1].set_xlabel('$t$'); axes[0, 1].set_ylabel('$x$')

# Absolute error
err_ac = np.abs(u_pinn_ac - u_fd_vis)
im2 = axes[1, 0].pcolormesh(T_vis.numpy(), X_vis.numpy(), err_ac,
                              cmap='hot_r', shading='auto')
plt.colorbar(im2, ax=axes[1, 0], label='$|u_{PINN} - u_{FD}|$')
axes[1, 0].set_title('Absolute error')
axes[1, 0].set_xlabel('$t$'); axes[1, 0].set_ylabel('$x$')

# Training loss curve
axes[1, 1].semilogy(loss_history_ac, lw=1.2, color='C3')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Total loss')
axes[1, 1].set_title('Training loss — Allen–Cahn PINN')

plt.tight_layout()
plt.show()

rel_l2_ac = (np.sqrt(np.mean(err_ac**2))
             / np.sqrt(np.mean(u_fd_vis**2) + 1e-12))
print(f'Relative L2 error (PINN vs FD): {rel_l2_ac:.4e}')

### 4.6 Comparison of solution profiles at selected times

In [ ]:
# Select four time snapshots for a side-by-side comparison
snapshot_fractions = [0.0, 0.33, 0.67, 1.0]
snapshot_indices   = [int(f * (Nt_vis - 1)) for f in snapshot_fractions]

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

for ax, idx in zip(axes, snapshot_indices):
    t_snap = t_fd_history[idx]
    ax.plot(x_fd, u_fd_vis[:, idx],     'k--', lw=2,   label='FD reference')
    ax.plot(x_fd, u_pinn_ac[:, idx],    'C0',  lw=1.8, label='PINN')
    ax.set_title(f'$t = {t_snap:.2f}$')
    ax.set_xlabel('$x$')
    ax.set_ylim(-1.3, 1.3)
    ax.axhline(0, color='grey', lw=0.6, ls=':')

axes[0].set_ylabel('$u(x, t)$')
axes[0].legend(fontsize=9)
plt.suptitle('Allen–Cahn solution profiles: PINN vs finite differences', y=1.01)
plt.tight_layout()
plt.show()

### 4.7 Discussion

The Allen–Cahn equation is a stiff, nonlinear PDE that challenges the PINN in several ways:

- **Nonlinearity**: the cubic term $u^3$ creates a competition between diffusion and reaction that can drive sharp interfaces. The PINN must learn this competition from collocation points alone.
- **Spectral bias**: as discussed in [Session 9](Session9.ipynb), the network preferentially learns low-frequency components of the solution. The sharp interface near $t = 0$ (caused by the initial condition) can be slow to resolve. Increasing the number of collocation points near $t = 0$ via adaptive sampling (see [Session 10](Session10.ipynb)) would improve accuracy.
- **Loss weighting**: the initial condition weight $\lambda_{\text{IC}} = 20$ was chosen to be large because the IC determines the entire subsequent evolution; without strong enforcement the solution drifts.
- **Comparison with FD**: the FD scheme is unconditionally stable (due to the implicit treatment of the diffusion term) and serves as a reliable reference. The PINN error tends to accumulate at later times where the solution has evolved furthest from the IC, which is consistent with the causal propagation difficulty identified in the PINN literature.

**Extensions.** A standard improvement for stiff reaction–diffusion PINNs is the use of **causal training** (Wang et al., 2022): the loss at time $t$ is weighted by $\exp(-c\,\mathcal{L}(t'))$ for earlier times $t' < t$, preventing the network from fitting late-time residuals before the early-time dynamics are resolved.

## 5. Comparison across the three applications

Having completed all three examples, it is instructive to compare them systematically.

| Feature | Schrödinger (TISE) | Stokes cavity | Allen–Cahn |
|---|---|---|---|
| **PDE type** | 2nd-order ODE (eigenvalue) | 2D Laplace | Nonlinear parabolic PDE |
| **Unknowns** | $\psi(x)$ and $E$ | $\omega(x,y)$ | $u(x,t)$ |
| **BCs** | Dirichlet, $\psi = 0$ | Mixed Dirichlet | Neumann |
| **Special challenge** | Sign/norm degeneracy | Non-homogeneous BC | Nonlinearity, stiffness |
| **PINN input dimension** | 1 | 2 | 2 |
| **Connection to earlier sessions** | [Session 8](Session8.ipynb) inverse problems | [Session 8](Session8.ipynb) Poisson equation | [Session 9](Session9.ipynb) spectral bias |
| **Validation method** | Analytic formula | Visual inspection | FD reference |

The table illustrates that the PINN recipe is robust across very different physical settings. The core steps — define the network, sample collocation points, compute residuals via automatic differentiation, minimise a multi-term loss — remain unchanged. What varies is the physical knowledge encoded in the residual and the appropriate choice of constraints and loss weights.

## 6. Summary and looking ahead

This session demonstrated PINNs applied to three distinct physical systems:

1. **Schrödinger equation**: the PINN simultaneously learnt the wave function and the eigenenergy by treating $E$ as a learnable parameter. A normalisation constraint was essential to avoid the trivial zero solution.

2. **Stokes cavity flow**: the Navier–Stokes equations in the creeping-flow limit reduce to a Laplace equation. The PINN required no architectural change from the Poisson solver in [Session 8](Session8.ipynb), only different boundary conditions.

3. **Allen–Cahn equation**: a nonlinear reaction–diffusion equation was solved and validated against a finite-difference reference. The stiffness of the problem highlighted the importance of strong initial condition enforcement and adaptive sampling.

**Key takeaways:**
- PINNs transfer to new physics by changing the residual, not the architecture.
- Constraints such as normalisation or sign-fixing are often necessary to resolve degeneracies.
- Validating against an analytic or numerical reference solution is essential practice.
- Stiff or sharply varying solutions require careful attention to collocation density and loss weights.

[Session 12](Session12.ipynb) is the capstone of this course. You will choose a physical system of your own interest and implement a complete PINN solver, drawing on all the techniques from Sessions 1–11. Suggested projects include: a pendulum with unknown damping, heat flow in an L-shaped domain, or the 2D wave equation with a non-trivial initial pulse. You will also compare your PINN solution quantitatively with a traditional numerical method and critically assess where PINNs offer advantages and where they fall short.

**Further reading:**
- Raissi, Perdikaris & Karniadakis (2019) *Physics-informed neural networks*, Journal of Computational Physics — the original PINN paper.
- Wang, Yu & Perdikaris (2022) *Respecting causality for training physics-informed neural networks*, Computer Methods in Applied Mechanics and Engineering — addresses the temporal stiffness challenge seen in the Allen–Cahn problem.
- Cai, Mao, Wang, Yin & Karniadakis (2021) *Physics-informed neural networks for heat transfer problems*, Journal of Heat Transfer — a concise treatment of PINNs applied to fluid and thermal systems.